# Manifold-Rank-Fusion: End-to-End Demo

This notebook runs the full pipeline once, end to end, on the sample data bundled
with this repository: the **Corel5k** dataset (5,000 images, 50 classes) with
pre-extracted **Swin Transformer** features.

Stages covered:

1. Load pre-extracted features
2. Neighbor embedding projection (UMAP)
3. Ranked list generation (Ball Tree)
4. Rank-based manifold learning re-ranking (CPRR)
5. Rank aggregation (Borda Count)
6. Post re-ranking
7. Effectiveness evaluation (MAP / Precision@k / Recall@k)
8. Bonus: comparing Borda Count against RRF and CombSUM

Sections 4, 6, and 7 call into [pyUDLF](https://github.com/UDLF/pyUDLF) (installed
from source -- see the main README) and require the UDLF binary it manages. Sections
2, 3, and 5 only need `numpy`, `scikit-learn`, and `umap-learn`.

For the same pipeline as standalone reusable scripts, see the numbered files in this
`examples/` folder and [examples/README.md](README.md).


## 1. Setup

In [1]:
import os
import sys

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # silence native TF/cuDNN init logging
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")  # avoid GPU-backend registration noise (CPU-only demo)

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from manifold_rank_fusion.aggregation import aggregate_batch
from manifold_rank_fusion.io_utils import read_ranked_lists, write_ranked_lists
from manifold_rank_fusion.projection import build_ranked_lists, project_features
from manifold_rank_fusion.udlf_rerank import DATASET_K, evaluate_ranking_file, run_rerank

DATASET = "corel5k"
DESCRIPTOR = "swintf"
SIZE_DATASET = 5000
TOP_K = 1000
K = DATASET_K[DATASET]  # rank-based re-ranking neighborhood size

DATA_DIR = os.path.join("..", "data", DATASET)
OUTPUT_DIR = os.path.join("..", "output", "demo")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURES_PATH = os.path.join(DATA_DIR, f"features_{DESCRIPTOR}_{DATASET}.npy")
LISTS_FILE = os.path.join(DATA_DIR, f"{DATASET}_lists.txt")
CLASSES_FILE = os.path.join(DATA_DIR, f"{DATASET}_classes.txt")

print(f"Dataset: {DATASET} | Descriptor: {DESCRIPTOR} | K = {K}")


E0000 00:00:1788290169.749810  127720 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788290169.753945  127720 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788290169.766374  127720 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788290169.766397  127720 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788290169.766400  127720 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788290169.766403  127720 computation_placer.cc:177] computation placer already registered. Please check linka

Dataset: corel5k | Descriptor: swintf | K = 100


## 2. Load Pre-Extracted Features

In [2]:
features = np.load(FEATURES_PATH)
print(f"Features shape: {features.shape}")


Features shape: (5000, 1024)


## 3. Neighbor Embedding Projection (UMAP)

The deep features are projected into a low-dimensional space with UMAP, using the
library's default hyperparameters (`n_components=2`, `n_neighbors=15`, `min_dist=0.1`,
Euclidean metric) and a fixed random seed for reproducibility.


In [3]:
umap_features = project_features(features, random_state=42)
print(f"UMAP projection shape: {umap_features.shape}")


UMAP projection shape: (5000, 2)


## 4. Ranked List Generation (Ball Tree)

A ranked list is generated for every query by ordering Euclidean distances with a
Ball Tree, once directly on the original features (the baseline ranking) and once on
the UMAP projection.


In [4]:
_, original_ranking = build_ranked_lists(features, top_k=TOP_K)
_, umap_ranking = build_ranked_lists(umap_features, top_k=TOP_K)

original_ranking_path = os.path.join(OUTPUT_DIR, f"{DATASET}_{DESCRIPTOR}.txt")
umap_ranking_path = os.path.join(OUTPUT_DIR, f"{DATASET}_{DESCRIPTOR}_umap.txt")

write_ranked_lists(original_ranking, original_ranking_path)
write_ranked_lists(umap_ranking, umap_ranking_path)

print("Original ranking, query 0, top 10:", original_ranking[0][:10])
print("UMAP ranking, query 0, top 10:     ", umap_ranking[0][:10])


Original ranking, query 0, top 10: [   0 4889 4444 4512 2222 4501 4823 3222 1889    1]
UMAP ranking, query 0, top 10:      [   0 4978 4512 4801 1111 3778 4878 4556 4523 4745]


## 5. Rank-Based Manifold Learning Re-Ranking (CPRR)

CPRR re-ranks the original-feature ranking by exploiting contextual/manifold
information (Cartesian Product of Ranking References). This step calls the UDLF
binary through pyUDLF.


In [5]:
cprr_output_path = os.path.join(OUTPUT_DIR, f"{DATASET}_{DESCRIPTOR}_CPRR")

run_rerank(
    method="CPRR",
    input_file=original_ranking_path,
    size_dataset=SIZE_DATASET,
    lists_file=LISTS_FILE,
    classes_file=CLASSES_FILE,
    k=K,
    output_file=True,
    output_file_path=cprr_output_path,
    output_log_file_path=os.path.join(OUTPUT_DIR, "log_CPRR.txt"),
)

cprr_ranking_path = cprr_output_path + ".txt"
cprr_ranking = read_ranked_lists(cprr_ranking_path, top_k=TOP_K)
print(f"CPRR re-ranking written to {cprr_ranking_path}")


[INFO] InputType initialized.


[INFO] UDLF binary and config found successfully.


[INFO] Running UDLF framework with config: /tmp/tmpipt5kg1w.ini


[INFO] UDLF run successfully.


[INFO] pyUDLF execution complete!


Time       =  0.8785 s
P@4        = {'Before': '0.9808', 'After': '0.9827', 'Gain': '0.1988%'}
P@5        = {'Before': '0.9773', 'After': '0.9809', 'Gain': '0.3684%'}
P@10       = {'Before': '0.9615', 'After': '0.9755', 'Gain': '1.4582%'}
P@15       = {'Before': '0.9493', 'After': '0.9738', 'Gain': '2.5758%'}
P@20       = {'Before': '0.9368', 'After': '0.9721', 'Gain': '3.7704%'}
P@30       = {'Before': '0.9146', 'After': '0.9697', 'Gain': '6.0342%'}
P@50       = {'Before': '0.8722', 'After': '0.9635', 'Gain': '10.4784%'}
P@100      = {'Before': '0.7127', 'After': '0.8742', 'Gain': '22.6643%'}
Recall@4   = {'Before': '0.0392', 'After': '0.0393', 'Gain': '0.1988%'}
Recall@5   = {'Before': '0.0489', 'After': '0.0490', 'Gain': '0.3684%'}
Recall@10  = {'Before': '0.0961', 'After': '0.0975', 'Gain': '1.4582%'}
Recall@20  = {'Before': '0.1874', 'After': '0.1944', 'Gain': '3.7704%'}
Recall@40  = {'Before': '0.3574', 'After': '0.3869', 'Gain': '8.2520%'}
MAP        = {'Before': '0.7392', 'Afte

CPRR re-ranking written to ../output/demo/corel5k_swintf_CPRR.txt


## 6. Rank Aggregation (Borda Count)

The UMAP-projection ranking and the CPRR re-ranking are combined with Borda Count --
the main aggregation strategy used by this framework. This step is pure Python/numpy
and does not require pyUDLF.


In [6]:
borda_ranking = aggregate_batch(umap_ranking, cprr_ranking, method="borda", top_k=TOP_K)

borda_ranking_path = os.path.join(OUTPUT_DIR, f"borda_{DATASET}_{DESCRIPTOR}_CPRR.txt")
write_ranked_lists(borda_ranking, borda_ranking_path)

print("Aggregated (Borda) ranking, query 0, top 10:", borda_ranking[0][:10])


Aggregated (Borda) ranking, query 0, top 10: [0, 4512, 4978, 4801, 3778, 4523, 1556, 4501, 4444, 4823]


## 7. Post Re-Ranking

An additional CPRR pass is applied to the aggregated ranking for a final refinement
step.


In [7]:
post_rerank_output_path = os.path.join(OUTPUT_DIR, f"borda_{DATASET}_{DESCRIPTOR}_CPRR_postCPRR")

run_rerank(
    method="CPRR",
    input_file=borda_ranking_path,
    size_dataset=SIZE_DATASET,
    lists_file=LISTS_FILE,
    classes_file=CLASSES_FILE,
    k=K,
    output_file=True,
    output_file_path=post_rerank_output_path,
    output_log_file_path=os.path.join(OUTPUT_DIR, "log_post_CPRR.txt"),
)

post_rerank_path = post_rerank_output_path + ".txt"
print(f"Post-re-ranked (final) ranking written to {post_rerank_path}")


[INFO] InputType initialized.


[INFO] UDLF binary and config found successfully.


[INFO] Running UDLF framework with config: /tmp/tmp2kzky3g5.ini


[INFO] UDLF run successfully.


[INFO] pyUDLF execution complete!


Time       =  0.6796 s
P@4        = {'Before': '0.9867', 'After': '0.9846', 'Gain': '-0.2128%'}
P@5        = {'Before': '0.9856', 'After': '0.9835', 'Gain': '-0.2110%'}
P@10       = {'Before': '0.9823', 'After': '0.9811', 'Gain': '-0.1222%'}
P@15       = {'Before': '0.9802', 'After': '0.9796', 'Gain': '-0.0680%'}
P@20       = {'Before': '0.9788', 'After': '0.9788', 'Gain': '0.0010%'}
P@30       = {'Before': '0.9763', 'After': '0.9782', 'Gain': '0.1933%'}
P@50       = {'Before': '0.9650', 'After': '0.9723', 'Gain': '0.7523%'}
P@100      = {'Before': '0.8561', 'After': '0.9375', 'Gain': '9.5110%'}
Recall@4   = {'Before': '0.0395', 'After': '0.0394', 'Gain': '-0.2128%'}
Recall@5   = {'Before': '0.0493', 'After': '0.0492', 'Gain': '-0.2110%'}
Recall@10  = {'Before': '0.0982', 'After': '0.0981', 'Gain': '-0.1222%'}
Recall@20  = {'Before': '0.1958', 'After': '0.1958', 'Gain': '0.0010%'}
Recall@40  = {'Before': '0.3889', 'After': '0.3905', 'Gain': '0.4140%'}
MAP        = {'Before': '0.9020', 

## 8. Effectiveness Evaluation

MAP and Precision@100 are computed at every stage of the pipeline -- the original
baseline, UMAP projection alone, CPRR re-ranking alone, Borda Count aggregation, and
the final post-re-ranked result -- to see how each stage contributes.


In [8]:
import re


def slugify(label):
    # pyUDLF's path validator rejects spaces/parentheses; keep the pretty
    # label for the table and a filesystem-safe slug for the log filename.
    return re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_")


def evaluate(path, label):
    log = evaluate_ranking_file(
        ranked_list_file=path,
        size_dataset=SIZE_DATASET,
        lists_file=LISTS_FILE,
        classes_file=CLASSES_FILE,
        top_k=TOP_K,
        output_log_file_path=os.path.join(OUTPUT_DIR, f"log_eval_{slugify(label)}.txt"),
    )
    return {
        "Stage": label,
        "MAP": float(log["MAP"]["After"]),
        "P@100": float(log["P@100"]["After"]),
    }

stages = [
    (original_ranking_path, "Baseline (original features)"),
    (umap_ranking_path, "UMAP only"),
    (cprr_ranking_path, "CPRR re-rank only"),
    (borda_ranking_path, "Borda aggregation (UMAP + CPRR)"),
    (post_rerank_path, "Borda aggregation + post re-rank (CPRR)"),
]

pipeline_results = [evaluate(path, label) for path, label in stages]


[INFO] InputType initialized.


[INFO] UDLF binary and config found successfully.


[INFO] Running UDLF framework with config: /tmp/tmpu5vob8z0.ini


[INFO] UDLF run successfully.


[INFO] pyUDLF execution complete!


...

[INFO] Running UDLF framework with config: /tmp/tmp90z99kio.ini


Time       =  0.0000 s
P@4        = {'Before': '0.9867', 'After': '0.9867', 'Gain': '0.0000%'}
P@5        = {'Before': '0.9856', 'After': '0.9856', 'Gain': '0.0000%'}
P@10       = {'Before': '0.9823', 'After': '0.9823', 'Gain': '0.0000%'}
P@15       = {'Before': '0.9802', 'After': '0.9802', 'Gain': '0.0000%'}
P@20       = {'Before': '0.9788', 'After': '0.9788', 'Gain': '0.0000%'}
P@30       = {'Before': '0.9763', 'After': '0.9763', 'Gain': '0.0000%'}
P@50       = {'Before': '0.9650', 'After': '0.9650', 'Gain': '0.0000%'}
P@100      = {'Before': '0.8561', 'After': '0.8561', 'Gain': '0.0000%'}
Recall@4   = {'Before': '0.0395', 'After': '0.0395', 'Gain': '0.0000%'}
Recall@5   = {'Before': '0.0493', 'After': '0.0493', 'Gain': '0.0000%'}
Recall@10  = {'Before': '0.0982', 'After': '0.0982', 'Gain': '0.0000%'}
Recall@20  = {'Before': '0.1958', 'After': '0.1958', 'Gain': '0.0000%'}
Recall@40  = {'Before': '0.3889', 'After': '0.3889', 'Gain': '0.0000%'}
MAP        = {'Before': '0.9020', 'After'

[INFO] UDLF run successfully.


[INFO] pyUDLF execution complete!


Time       =  0.0000 s
P@4        = {'Before': '0.9846', 'After': '0.9846', 'Gain': '0.0000%'}
P@5        = {'Before': '0.9835', 'After': '0.9835', 'Gain': '0.0000%'}
P@10       = {'Before': '0.9811', 'After': '0.9811', 'Gain': '0.0000%'}
P@15       = {'Before': '0.9796', 'After': '0.9796', 'Gain': '0.0000%'}
P@20       = {'Before': '0.9788', 'After': '0.9788', 'Gain': '0.0000%'}
P@30       = {'Before': '0.9782', 'After': '0.9782', 'Gain': '0.0000%'}
P@50       = {'Before': '0.9723', 'After': '0.9723', 'Gain': '0.0000%'}
P@100      = {'Before': '0.9375', 'After': '0.9375', 'Gain': '0.0000%'}
Recall@4   = {'Before': '0.0394', 'After': '0.0394', 'Gain': '0.0000%'}
Recall@5   = {'Before': '0.0492', 'After': '0.0492', 'Gain': '0.0000%'}
Recall@10  = {'Before': '0.0981', 'After': '0.0981', 'Gain': '0.0000%'}
Recall@20  = {'Before': '0.1958', 'After': '0.1958', 'Gain': '0.0000%'}
Recall@40  = {'Before': '0.3905', 'After': '0.3905', 'Gain': '0.0000%'}
MAP        = {'Before': '0.9539', 'After'

In [9]:
pipeline_summary = pd.DataFrame(pipeline_results).set_index("Stage")
pipeline_summary


,MAP,P@100
Stage,,
Baseline (original features),0.7392,0.7127
UMAP only,0.9392,0.9315
CPRR re-rank only,0.8746,0.8742
Borda aggregation (UMAP + CPRR),0.9020,0.8561
Borda aggregation + post re-rank (CPRR),0.9539,0.9375


## 9. Bonus: Comparing Aggregation Strategies (Borda Count vs. RRF vs. CombSUM)

The same UMAP ranking and CPRR re-ranking are aggregated with all three fusion
strategies implemented in this framework, for a direct comparison.


In [10]:
aggregation_methods = ["borda", "rrf", "combsum"]
comparison_results = []

for method in aggregation_methods:
    method_kwargs = {"k": 60} if method == "rrf" else {}
    aggregated = aggregate_batch(
        umap_ranking, cprr_ranking, method=method, top_k=TOP_K, **method_kwargs
    )

    path = os.path.join(OUTPUT_DIR, f"{method}_{DATASET}_{DESCRIPTOR}_CPRR.txt")
    write_ranked_lists(aggregated, path)

    comparison_results.append(evaluate(path, method.upper()))


[INFO] InputType initialized.


[INFO] UDLF binary and config found successfully.


[INFO] Running UDLF framework with config: /tmp/tmp084lmee5.ini


[INFO] UDLF run successfully.


[INFO] pyUDLF execution complete!


...

[INFO] UDLF binary and config found successfully.


[INFO] Running UDLF framework with config: /tmp/tmpyq24pspq.ini


[INFO] UDLF run successfully.


[INFO] pyUDLF execution complete!


Time       =  0.0000 s
P@4        = {'Before': '0.9843', 'After': '0.9843', 'Gain': '0.0000%'}
P@5        = {'Before': '0.9819', 'After': '0.9819', 'Gain': '0.0000%'}
P@10       = {'Before': '0.9776', 'After': '0.9776', 'Gain': '0.0000%'}
P@15       = {'Before': '0.9754', 'After': '0.9754', 'Gain': '0.0000%'}
P@20       = {'Before': '0.9742', 'After': '0.9742', 'Gain': '0.0000%'}
P@30       = {'Before': '0.9720', 'After': '0.9720', 'Gain': '0.0000%'}
P@50       = {'Before': '0.9673', 'After': '0.9673', 'Gain': '0.0000%'}
P@100      = {'Before': '0.9060', 'After': '0.9060', 'Gain': '0.0000%'}
Recall@4   = {'Before': '0.0394', 'After': '0.0394', 'Gain': '0.0000%'}
Recall@5   = {'Before': '0.0491', 'After': '0.0491', 'Gain': '0.0000%'}
Recall@10  = {'Before': '0.0978', 'After': '0.0978', 'Gain': '0.0000%'}
Recall@20  = {'Before': '0.1948', 'After': '0.1948', 'Gain': '0.0000%'}
Recall@40  = {'Before': '0.3880', 'After': '0.3880', 'Gain': '0.0000%'}
MAP        = {'Before': '0.9428', 'After'

In [11]:
comparison_summary = pd.DataFrame(comparison_results).set_index("Stage")
comparison_summary


,MAP,P@100
Stage,,
BORDA,0.9020,0.8561
RRF,0.9344,0.8912
COMBSUM,0.9428,0.9060


## Summary

This notebook walked through the complete pipeline on a single dataset/descriptor
pair. To run it on other datasets or descriptors, reuse the standalone scripts in
this folder (see [examples/README.md](README.md)) -- each pipeline stage
(`projection.py`, `udlf_rerank.py`, `aggregation.py`) is also available as a
importable function in `manifold_rank_fusion` for scripting a full experiment sweep.
